# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [1]:
# imports
import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI
import gradio as gr # oh yeah!

In [2]:
# constants

MODEL_GPT = 'openai/gpt-4o-nano'
MODEL_LLAMA = 'llama3.2'

In [4]:
# set up environment
#connecting to openrouter
load_dotenv(override=True)
api_key = os.getenv('OPENROUTER_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-or-v1"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)

API key found and looks good so far!


In [6]:
def chat_with_model(user_message, model_select):
    
    system_prompt = f"""
You are a teacher who sets a tough exam for school children from any given text.
Develop 2 questions from the given text along with 1 line answers that are expected from students.
"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message},
    ]

    # If using GPT (OpenRouter)
    if model_select == "GPT":
        response = client.chat.completions.create(
            model=MODEL_GPT,
            messages=messages,
            temperature=0.3,
        )
        return response.choices[0].message.content

    # If using LLaMA (Ollama)
    elif model_select == "LLaMA":
        import ollama
        response = ollama.chat(
            model=MODEL_LLAMA,
            messages=messages,
        )
        return response["message"]["content"]

In [7]:
with gr.Blocks() as demo:
    gr.Markdown("## Exam questions with Answers")

    user_input = gr.Textbox(
        label="Ask your question",
        lines=6,
        placeholder="Paste text from which you want to generate questions and answers..."
    )

    model_dropdown = gr.Dropdown(
        choices=["GPT", "LLaMA"],
        value="GPT",
        label="Select Model"
    )


    output = gr.Markdown(label="Response")

    submit_btn = gr.Button("Quiz me")

    submit_btn.click(
        fn=chat_with_model,
        inputs=[user_input, model_dropdown],
        outputs=output,
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


I pasted : The Indian early medieval age, from 600 to 1200 CE, was defined by regional kingdoms and cultural diversity. No ruler of this period was able to create an empire and consistently control lands much beyond their core region. During this time, pastoral peoples, whose land had been cleared to make way for the growing agricultural economy, were accommodated within caste society, as were new non-traditional ruling classes. The caste system consequently began to show regional differences.

Response I got: Question 1 What was the main characteristic of the Indian early medieval age?

Expected answer: Regional kingdoms and cultural diversity.

Question 2 How did the caste system change during the Indian early medieval age?

Expected answer: It began to show regional differences.